In [6]:
# Install packages
# %pip install requests
# %pip install bs4
# %pip install googlesearch-python


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
  Preparing metadata (setup.py) ... done
  Created wheel for googlesearch-python: filename=googlesearch_python-1.2.3-py3-none-any.whl size=4229 sha256=aabf0357a49a6a798fd985e3327f817a844deef28704a6f70017547b4043a8ab
  Stored in directory: /Users/modekurt/Library/Caches/pip/wheels/be/78/a3/d80e85ac9551489cd9c44f59a0bc5972e79e680ce9cc6055ca
Successfully built googlesearch-python
Note: you may need to restart the kernel to use updated packages.


In [16]:
import requests
from bs4 import BeautifulSoup
from googlesearch import search
import os
import pandas as pd
import numpy

In [87]:
planpath = 'plans/'

In [86]:
#Now search on google for climate action plans using extracted link

def download_search_results(query, city_name, city_code, country_iso, num_results):
    # Perform Google search and get the first 'num_results' URLs
    # sleep_interval=2
    search_results = list(search(query, num_results=num_results))
    print(search_results)

    # Create a directory to save the PDFs
    folder_path = planpath+country_iso+'/'+city_code+'-'+city_name
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

    #collect info on downloads
    links, names = [], []

    # Download content from each URL
    for i, url in enumerate(search_results):
        try:
            response = requests.get(url)

            # Save the content to a file (you can modify this part based on your needs)
            if url.endswith('.pdf'):
                pdf_name = os.path.join(folder_path, city_code+'-[date]-'+os.path.basename(url))

                with open(pdf_name, 'wb') as pdf_file:
                    pdf_file.write(response.content)
                    print(f"Downloaded result {i + 1}: {url}")

                    links.append(url)
                    names.append(pdf_name)
        except Exception as e:
            print(f"Error downloading result {i + 1}: {url}\nError: {e}")

    return links, names

# Example usage
# search_query = 'Mendoza' + 'climate adaptation plan' + 'filetype:pdf'
# download_search_results(search_query, 'sacramento', '12345', num_results=1)

In [92]:
#Automate search of plans for all cities 

#load city list and list of searches in multiple languages 
city_list = pd.read_csv('output/capital_and_1mill_pop_cities-mod.csv')
lang_list = pd.read_csv('data/Multilingual Search List.csv')
type_a_search_en = 'climate|"climate change" adaptation|protection|resilience plan|strategy|roadmap|program|measures|framework'
results_num = 1 #number of search results we want to return

city_codes, plan_links, plan_names = [], [], []

def plan_search(row):
    city_lang = str(row['Language'])
    city_lang_code = str(row['Language Code'])
    city_name = row['City']
    country_iso = row['ISO_Code']
    
    lang_search = lang_list.loc[lang_list['Code'] == city_lang_code, 'Plan Type A Search (adaptation plans)']
    #if the language doesn't exist in our search list, set the search to english
    type_a_search =  type_a_search_en if lang_search.isnull().any() else lang_search.to_string(index=False)
    query = city_name + ' ' + type_a_search + ' filetype:pdf'
    print(query)
    city_code = str(row['City Code'])
        
    #Download searches for city
    links, names = download_search_results(query, city_name, city_code, country_iso, results_num)
    # links, names = ["test", "test"], ["b", "bla"]
    
    
    city_codes.extend([city_code]*len(links))
    plan_links.extend(links)
    plan_names.extend(names)
    #Maybe add Document Status, Document Link (in Google Drive)?
    return 

#look for searches for each city
try:
    searched_plans = city_list.apply(plan_search, axis=1)
except Exception as e:
    print(f'Error: {e} ')

downloaded_plans = pd.DataFrame({'City Code': city_codes, 'Plan Link': plan_links, 'Plan Name': plan_names})

#Combine downloaded information with other relevant info like country, city name, etc
downloaded_plans['City Code'] = downloaded_plans['City Code'].astype(int)
downloaded_plans['Dataset Source'] = 'Automated Google Search'

downloaded_plans = downloaded_plans.merge(city_list, on='City Code')
downloaded_plans = downloaded_plans[['Country', 'City', 'City Code', 'ISO_Code', 'Plan Name', 'Plan Link', 'Dataset Source']]
#write output to file
downloaded_plans.to_csv("output/downloaded_plans.csv")

Adelaide climate|"climate change" adaptation|protection|... filetype:pdf
Error: 429 Client Error: Too Many Requests for url: https://www.google.com/sorry/index?continue=https://www.google.com/search%3Fq%3DAdelaide%252Bclimate%257C%2522climate%252Bchange%2522%252Badaptation%257Cprotection%257C...%252Bfiletype%253Apdf%26num%3D3%26hl%3Den%26start%3D0&hl=en&q=EhAqAoEKC8AQ5hkVIQqPZwTUGOystrAGIjApJgkri0l9HAIEMq4qCKh__9dlO7PMT18lAOYOtsU7vNL5YU3KY44IP5gePJP49NsyAXJaAUM 
